In [ ]:
# Train model

!python tools/export/train.py \
  --config config.yml

In [ ]:
# Train model checkpoint

!python tools/export/train.py \
  --config config.yml
  --checkpoint checkpoints/best.pt

In [ ]:
# Export model to ONNX

!python tools/export/export_onnx.py \
  --checkpoint /home/hellodev/Desktop/THAOCR/output/thaonet_rec/best.pt \
  --output model.onnx 

Using config: config.yml (target_h=64)
Loading model from /home/hellodev/Desktop/THAOCR/output/thaonet_rec/best.pt...
/home/hellodev/miniconda3/envs/thaocr/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
Traceback (most recent call last):
  File "/media/hellodev/3241b1af-b185-4d24-b5bc-5384221b1cb3/thaocr/tools/export/export_onnx.py", line 144, in <module>
    main()
  File "/media/hellodev/3241b1af-b185-4d24-b5bc-5384221b1cb3/thaocr/tools/export/export_onnx.py", line 96, in main
    model, vocab = load_model(args.checkpoint, device="cpu", config_path=config_path)
  File "/media/hellodev/3241b1af-b185-4d24-b5bc-5384221b1cb3/thaocr/tools/train/evaluate.py", line 42, in load_model
    model.load_state_dict(ckpt["model_state_dict"])
KeyError: 'model_state_dict'


In [ ]:
# Predict with ONNX

!python tools/export/predict_onnx.py \
  --model model90k/model.onnx \
  --vocab model90k/model_vocab.json \
  --image input_image \
  --height 32


Result: ពីព្រោះមនខ្មែរស្ថិតនៅក្នុងក្រុមមនុស្សឥណ្ឌូនេស៊ីយាង


In [6]:
pip install onnxruntime-gpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 MB 7.4 MB/s  0:00:40m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 7.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 6.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [onnxruntime-gpu] [onnxruntime-gpu]
Note: you may need to restart the kernel to use updated packages.


In [4]:
input_image = "/home/thareah/Desktop/senhay_dataset/khmer-hanuman-100k/data/images1/img_000056.png"

In [7]:
# Raw predict

!python tools/export/predict.py \
  --checkpoint /home/thareah/Desktop/THOr/latest.pt \
  --config config.yml \
  --image $input_image

Loading model from /home/thareah/Desktop/THOr/latest.pt...
Predicting 1 images...

                          img_000056.png  →  កលក់ខុនដូនៅទីតាំងរប


In [ ]:
# Raw predict

!python tools/export/predict.py \
  --checkpoint /home/thareah/Desktop/model/seanhay_ds/latest.pt \
  --config config.yml \
  --image input_image

Loading model from /home/thareah/Desktop/model/seanhay_ds/latest.pt...
Predicting 1 images...

                     train_000000001.png  →  ០០លានដុល្លារ ទៅក្នុងសេវាកម្មបង់ប្រាក់រ


In [ ]:
# Predict with ONNX

!python tools/export/predict_onnx.py \
  --model model90k/model.onnx \
  --vocab model90k/model_vocab.json \
  --image input_image \
  --height 32


Result: ០០លានដុល្លារ.ទៅក្នងសេវាកម្មបង់ប្រាក់រ


In [1]:
import os, random, shutil

# ===== CONFIG =====
SOURCE_DIR = "khmer-ocr-synth-v1"
ALL_TXT    = os.path.join(SOURCE_DIR, "train.txt")

OUT_DIR    = "output100k/"   # writable
NUM_TRAIN  = 89925
NUM_VAL    = 10076
SEED       = 42
# ==================

random.seed(SEED)

train_dir = os.path.join(OUT_DIR, "train")
val_dir   = os.path.join(OUT_DIR, "val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

with open(ALL_TXT, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total images:", len(lines))

random.shuffle(lines)
selected = lines[:NUM_TRAIN + NUM_VAL]
train_lines = selected[:NUM_TRAIN]
val_lines   = selected[NUM_TRAIN:]

def safe_copy(rel_path, dst_dir):
    # rel_path like: "train/img_....jpg"
    src_path = os.path.join(SOURCE_DIR, rel_path)
    if not os.path.exists(src_path):
        return None, src_path  # missing
    filename = os.path.basename(rel_path)
    dst_path = os.path.join(dst_dir, filename)
    shutil.copy2(src_path, dst_path)
    return filename, None

# ===== TRAIN =====
new_train_txt = []
missing_train = 0
for line in train_lines:
    rel_path, text = line.strip().split("\t", 1)
    filename, missing_src = safe_copy(rel_path, train_dir)
    if missing_src:
        missing_train += 1
        continue
    new_train_txt.append(f"train/{filename}\t{text}\n")

# ===== VAL =====
new_val_txt = []
missing_val = 0
for line in val_lines:
    rel_path, text = line.strip().split("\t", 1)
    filename, missing_src = safe_copy(rel_path, val_dir)
    if missing_src:
        missing_val += 1
        continue
    new_val_txt.append(f"val/{filename}\t{text}\n")

with open(os.path.join(OUT_DIR, "train.txt"), "w", encoding="utf-8") as f:
    f.writelines(new_train_txt)

with open(os.path.join(OUT_DIR, "val.txt"), "w", encoding="utf-8") as f:
    f.writelines(new_val_txt)

print("Done!")
print("Train:", len(new_train_txt), "missing:", missing_train)
print("Val:", len(new_val_txt), "missing:", missing_val)
print("Output folder:", OUT_DIR)

Total images: 338866
Done!
Train: 89925 missing: 0
Val: 10076 missing: 0
Output folder: output100k/
